# Semana 4 · ¿Por qué cambian las constelaciones con las estaciones?

Este cuaderno acompaña a la sesión de proyecto de la semana 4: calcular qué parte del cielo está mejor situada cada noche del año y dibujarlo como una gráfica periódica.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar.

Tiempo estimado: 30-45 minutos.

## 1. El operador resto y las funciones periódicas

La ascensión recta vuelve a empezar cada 24 horas: 25 h es lo mismo que 1 h. La operación que expresa eso es el **resto** de la división, que en Python se escribe `%`.

In [ ]:
print(26 % 24)      # 2: 26 h equivale a 2 h
print(-3 % 24)      # 21: en Python el resultado de % con divisor positivo nunca es negativo
print(24.5 % 24)    # 0.5

**Ejercicio.** Calcula 50 h mod 24 y −30 h mod 24 a mano, y comprueba con Python.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
print(50 % 24, -30 % 24)

## 2. Dónde está el Sol en el cielo, día a día

El Sol recorre la esfera celeste una vuelta completa al año, así que su ascensión recta aumenta 24 h en 365,24 días, es decir, unos 4 minutos al día. Tomando el equinoccio de marzo (día 80, AR = 0 h) como origen, una aproximación sencilla es

$$\alpha_\odot(d) \approx (d - 80)\cdot\frac{24}{365{,}24}\ \text{h} \pmod{24}$$

Ignora la forma elíptica de la órbita y la oblicuidad: el error es de unos 0,3 h como máximo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def ar_sol(dia):
    """Ascensión recta aproximada del Sol, en horas. dia: día del año (1 de enero = 1)."""
    return ((dia - 80) * 24 / 365.24) % 24

for nombre, dia in [("equinoccio de marzo", 80), ("solsticio de junio", 172), ("equinoccio de septiembre", 266), ("solsticio de diciembre", 355)]:
    print(f"{nombre:26s} AR del Sol ≈ {ar_sol(dia):5.2f} h")

Los valores reales son 0 h, 6 h, 12 h y 18 h. La aproximación acierta con una décima de hora.

## 3. Qué hay en el meridiano a cualquier hora

La estrella que cruza el meridiano en un instante tiene la ascensión recta del punto del cielo que está sobre tu cabeza (en dirección norte-sur). A las **12:00 solares** el Sol está en el meridiano, así que esa AR es la del Sol. Cada hora que pasa, el meridiano avanza 1 h de AR. Luego, a una hora solar $t$ (de 0 a 24):

$$\alpha_{\text{meridiano}} = \alpha_\odot + (t - 12)\ \text{h}$$

A la medianoche solar, $t = 24$, y el meridiano tiene $\alpha_\odot + 12$ h. Atención: la hora solar no es la del reloj. En España, con el horario de verano, la medianoche solar cae hacia las 02:00 del reloj.

In [ ]:
def ar_meridiano(dia, hora_solar=24):
    return (ar_sol(dia) + (hora_solar - 12)) % 24

inicio_mes = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
meses = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]

for m, d in zip(meses, inicio_mes):
    print(f"1 de {m}: AR en el meridiano a las 22 h solares {ar_meridiano(d, 22):5.2f} h   a medianoche solar {ar_meridiano(d, 24):5.2f} h")

## 4. La gráfica periódica

Representamos la AR del meridiano a medianoche a lo largo del año junto a la AR de algunas estrellas brillantes. Cuando la curva pasa por la línea de una estrella, esa estrella está en el meridiano a medianoche, lo mejor posible para observarla.

In [ ]:
# (nombre, AR en horas, aproximada al minuto; valores sin contrastar con un catálogo)
estrellas = [("Aldebarán", 4.60), ("Rigel", 5.25), ("Betelgeuse", 5.92), ("Sirio", 6.75),
             ("Régulo", 10.13), ("Spica", 13.42), ("Arturo", 14.27), ("Antares", 16.48),
             ("Vega", 18.62), ("Altair", 19.85), ("Deneb", 20.68)]

dias = np.arange(1, 366)
y = np.array([ar_meridiano(d, 24) for d in dias])
y_plot = y.copy()
y_plot[np.where(np.diff(y) < 0)[0] + 1] = np.nan     # corta la línea donde la AR vuelve a 0 h

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dias, y_plot, lw=2, label="AR en el meridiano a medianoche solar")
for nombre, ar in estrellas:
    ax.axhline(ar, color="gray", lw=0.7, ls="--")
    ax.text(366, ar, nombre, va="center", fontsize=9)
ax.set_xticks(inicio_mes)
ax.set_xticklabels(meses)
ax.set_ylim(0, 24)
ax.set_xlim(1, 365)
ax.set_ylabel("ascensión recta (h)")
ax.set_title("La parte del cielo que culmina a medianoche cambia a lo largo del año")
ax.legend(loc="upper left")
plt.subplots_adjust(right=0.85)
plt.show()

La curva es una recta que sube 24 h en un año y vuelve a empezar: una función **periódica** de periodo un año.

## 5. Qué estrellas están bien situadas cada mes

Una estrella está bien situada si su AR difiere en menos de 1,5 h de la del meridiano. Hay que tener en cuenta que la diferencia entre 23 h y 1 h son 2 h, no 22: usamos el resto para medirla.

In [ ]:
def diferencia(a, b):
    """Diferencia mínima entre dos ascensiones rectas, en horas (de 0 a 12)."""
    d = (a - b) % 24
    return min(d, 24 - d)

for m, d in zip(meses, inicio_mes):
    mer = ar_meridiano(d, 24)
    cerca = [n for n, ar in estrellas if diferencia(ar, mer) <= 1.5]
    print(f"1 de {m}: meridiano a medianoche {mer:5.2f} h -> {', '.join(cerca) if cerca else '(ninguna de la lista)'}")

## 6. Para pensar

1. Cambia el 24 por 21 en las llamadas a `ar_meridiano(d, 24)` de la última celda (anochecer a finales de otoño). ¿Qué estrellas están bien situadas el 1 de noviembre?
2. ¿Por qué el resultado se repite cada año y no cada 12 meses exactos? (Pista: año tropical frente a año sidéreo.)
3. Si un amigo en el hemisferio sur mira al mismo meridiano a la misma hora, ¿ve las mismas estrellas?